In [1]:
# Colab: install dependencies (run once)
# Note: Colab often already has torch; this should be safe.
!pip install -q sentence-transformers faiss-cpu google-genai termcolor scikit-learn pandas

# Prevent TensorFlow from being imported by transformers
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TORCH"] = "1"

print("Installed packages and set environment flags.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 70.4 MB/s eta 0:00:00
Installed packages and set environment flags.


In [2]:
# Secure API key input
from getpass import getpass
import os, json

# Set GENAI_MOCK_MODE True to demo without calling Gemini (recommended while testing)
GENAI_MOCK_MODE = False

if not GENAI_MOCK_MODE:
    key = getpass("Paste your GOOGLE_API_KEY (input hidden): ")
    os.environ["GOOGLE_API_KEY"] = key
else:
    print("GENAI_MOCK_MODE is ON. No external API calls will be made.")


Paste your GOOGLE_API_KEY (input hidden): ··········


In [3]:
from google.colab import files
uploaded = files.upload()  # choose the wiki_movie_plots_deduped.csv
# After upload, note the filename printed; will use the first uploaded file:
csv_path = list(uploaded.keys())[0]
print("Using uploaded file:", csv_path)

Saving wiki_movie_plots_deduped.csv to wiki_movie_plots_deduped.csv
Using uploaded file: wiki_movie_plots_deduped.csv


In [4]:
# ----------------------------
# Full RAG pipeline
# ----------------------------
import os, json
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import HTML, display
from termcolor import colored

In [5]:
# import genai, but tolerate absence in mock mode
try:
    from google import genai
except Exception:
    genai = None

In [6]:
# Config
CHUNK_SIZE_WORDS = 300
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
FAISS_TOP_K = 3

In [7]:
# utility: pretty JSON
def show_json(json_text):
    try:
        obj = json.loads(json_text)
        pretty = json.dumps(obj, indent=2, ensure_ascii=False)
    except Exception:
        pretty = str(json_text)
    html = "<pre style='background:#1e1e1e;color:#dcdcdc;padding:16px;border-radius:8px;font-size:12px;'>" + pretty + "</pre>"
    display(HTML(html))

def cprint(msg, color="cyan"):
    print(colored(msg, color))

In [8]:
# 1) load CSV
def load_data(path, max_rows=400):
    df = pd.read_csv(path).head(max_rows)
    df = df.loc[:, ["Title", "Plot"]].dropna().reset_index(drop=True)
    cprint(f"Loaded {len(df)} rows", "green")
    return df

In [9]:
# 2) chunking
def chunk_text(text, chunk_size_words=CHUNK_SIZE_WORDS):
    words = text.split()
    return [" ".join(words[i:i+chunk_size_words]).strip() for i in range(0, len(words), chunk_size_words) if words[i:i+chunk_size_words]]

def build_chunks(df):
    chunks, meta = [], []
    for _, row in df.iterrows():
        for ch in chunk_text(row["Plot"]):
            chunks.append(ch)
            meta.append({"title": row["Title"], "chunk": ch})
    cprint(f"Built {len(chunks)} chunks", "green")
    return chunks, meta


In [10]:
# 3) embeddings
def load_embedding_model(name=EMBEDDING_MODEL):
    cprint(f"Loading embedding model: {name}", "yellow")
    m = SentenceTransformer(name)
    cprint("Embedding model ready.", "green")
    return m

def embed_chunks(model, chunks, batch_size=64):
    emb = model.encode(chunks, convert_to_numpy=True, show_progress_bar=True, batch_size=batch_size)
    emb = np.array(emb, dtype="float32")
    cprint(f"Embeddings shape: {emb.shape}", "green")
    return emb

In [11]:
# 4) FAISS
def build_faiss_index(emb):
    dim = emb.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(emb)
    cprint(f"FAISS index built. N={index.ntotal}", "green")
    return index

In [12]:
# 5) retrieval
def retrieve(query, model, index, meta, top_k=FAISS_TOP_K):
    q_emb = model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = index.search(q_emb, top_k)
    hits = []
    for idx in indices[0]:
        if idx < 0: continue
        hits.append(meta[idx])
    return hits

In [13]:
# 6) genai wrapper
import json
import inspect

def get_genai_client():
    """
    Returns genai.Client(api_key=...) or raises helpful error.
    """
    if GENAI_MOCK_MODE:
        return None
    if genai is None:
        raise RuntimeError("google-genai package not available. Install it or enable GENAI_MOCK_MODE.")
    key = os.environ.get("GOOGLE_API_KEY", None)
    if not key:
        raise ValueError(
            "Missing GOOGLE_API_KEY. In Colab use:\n"
            "from getpass import getpass\n"
            "import os\n"
            "os.environ['GOOGLE_API_KEY'] = getpass('Paste API key (hidden): ')\n"
            "Then re-run the cell."
        )
    return genai.Client(api_key=key)

def _call_genai_with_fallback(client, **kwargs):
    """
    Try several method names across genai SDK versions.
    kwargs should include model, prompt/contents, max_output_tokens / max_tokens etc.
    Returns response object.
    """
    models_obj = getattr(client, "models", None)
    # Candidate method names & argument mapping
    candidates = []

    # modern doc example: client.models.generate_content(model=..., contents=...)
    candidates.append(("models.generate_content", {"model": kwargs.get("model"), "contents": kwargs.get("prompt")}))
    # some versions: client.models.generate(model=..., prompt=..., max_output_tokens=...)
    candidates.append(("models.generate", {"model": kwargs.get("model"), "prompt": kwargs.get("prompt"), "max_output_tokens": kwargs.get("max_output_tokens")}))
    # some older patterns: client.generate(model=..., prompt=...)
    candidates.append(("generate", {"model": kwargs.get("model"), "prompt": kwargs.get("prompt"), "max_output_tokens": kwargs.get("max_output_tokens")}))

    # Try candidates
    for name, call_kwargs in candidates:
        if name.startswith("models.") and models_obj is not None:
            method_name = name.split(".", 1)[1]
            if hasattr(models_obj, method_name):
                method = getattr(models_obj, method_name)
                try:
                    return method(**{k:v for k,v in call_kwargs.items() if v is not None})
                except TypeError:
                    # some method signatures may differ; attempt to prune unknown kwargs
                    sig = inspect.signature(method)
                    filtered = {k:v for k,v in call_kwargs.items() if k in sig.parameters and v is not None}
                    return method(**filtered)
                except Exception as e:
                    # try next fallback
                    last_exc = e
                    continue
        else:
            # top-level client.generate
            if hasattr(client, name):
                method = getattr(client, name)
                try:
                    return method(**{k:v for k,v in call_kwargs.items() if v is not None})
                except TypeError:
                    sig = inspect.signature(method)
                    filtered = {k:v for k,v in call_kwargs.items() if k in sig.parameters and v is not None}
                    return method(**filtered)
                except Exception as e:
                    last_exc = e
                    continue
    # If all attempts fail, raise last encountered exception (if any) or a clear error
    raise RuntimeError("Unable to call genai model generate method; tried common method names. "
                       "Please check your google-genai version. Last error: {}".format(repr(locals().get('last_exc', None))))

def generate_answer(query, contexts, model_name="gemini-2.5-flash", max_output_tokens=400):
    """
    Robust generate_answer: works across genai SDK versions and supports mock mode.
    Returns a JSON string with {answer, contexts, reasoning}.
    """
    # Build context text
    ctx_text = "\n\n".join([f"{c['title']}: {c['chunk']}" for c in contexts])
    prompt = (
        "You are a movie RAG assistant. Answer the question using ONLY the retrieved plot snippets.\n"
        "Be concise. Return ONLY valid JSON with fields: answer (string), contexts (array of snippets), reasoning (string).\n\n"
        "Question:\n" + query + "\n\n"
        "Retrieved Context:\n" + ctx_text + "\n\n"
        "Output JSON only."
    )

    # Mock mode: deterministic offline response (useful for demo/recording)
    if GENAI_MOCK_MODE:
        mock = {
            "answer": "Mock: Gemini call skipped (GENAI_MOCK_MODE=True). Use the contexts to find the likely movie.",
            "contexts": [c["chunk"] for c in contexts],
            "reasoning": "Mock mode: returning canned JSON for reliable demos without network calls."
        }
        return json.dumps(mock, ensure_ascii=False)

    # Live mode: get client and call with fallback
    client = get_genai_client()
    try:
        resp = _call_genai_with_fallback(client, model=model_name, prompt=prompt, max_output_tokens=max_output_tokens)
    except Exception as e:
        # Provide helpful actionable error
        raise RuntimeError(
            "Failed to call genai model. Error: {}\n"
            "Check your google-genai package version and that GOOGLE_API_KEY is set.".format(e)
        )

    # Extract text from response robustly
    text_out = None
    if hasattr(resp, "text") and resp.text:
        text_out = resp.text
    else:
        # Some SDKs nest content differently
        # Try to look for .response, .output or .content fields
        try:
            # convert object -> str as last resort
            text_out = str(resp)
        except Exception:
            text_out = ""

    # Attempt to parse JSON. If model returned a JSON string, parse and return canonical JSON.
    try:
        parsed = json.loads(text_out)
        return json.dumps(parsed, ensure_ascii=False)
    except Exception:
        # Not strict JSON: wrap the raw text into the JSON structure
        fallback = {
            "answer": text_out.strip(),
            "contexts": [c["chunk"] for c in contexts],
            "reasoning": "Model output wasn't strict JSON; returned raw model text under 'answer'."
        }
        return json.dumps(fallback, ensure_ascii=False)


In [14]:
# 7) run RAG
def show_results(results):
    return pd.DataFrame(results)

def run_rag(query, df, emb_model, faiss_index, meta, top_k=FAISS_TOP_K):
    cprint(f"Query: {query}", "cyan")
    hits = retrieve(query, emb_model, faiss_index, meta, top_k=top_k)
    if hits:
        display(show_results(hits))
    else:
        cprint("No hits found.", "red")
    out = generate_answer(query, hits)
    show_json(out)
    return out

In [15]:
# 8) simple evaluation
EVAL_SET = [
    {"query": "Which movie has an AI that becomes dangerous?", "expected_keywords": ["HAL", "HAL 9000", "2001", "Space Odyssey"]}
]
def evaluate_rag(emb_model, faiss_index, meta, eval_set=EVAL_SET):
    rows = []
    for item in eval_set:
        q = item["query"]; expected = item.get("expected_keywords", [])
        retrieved = retrieve(q, emb_model, faiss_index, meta, top_k=5)
        combined = " ".join([r["chunk"] for r in retrieved]).lower()
        hit = any(k.lower() in combined for k in expected) if expected else None
        q_emb = emb_model.encode([q], convert_to_numpy=True)
        if retrieved:
            r_emb = emb_model.encode([r["chunk"] for r in retrieved], convert_to_numpy=True)
            max_sim = float(cosine_similarity(q_emb, r_emb).max())
        else:
            max_sim = 0.0
        rows.append({"query": q, "hit_rate": bool(hit), "max_similarity": round(max_sim, 4)})
    return pd.DataFrame(rows)

### **Demo execution cell**

In [16]:
# Make sure csv_path variable is set from upload or Drive mount step
assert 'csv_path' in globals(), "Set csv_path from the upload cell (files.upload or Drive mount)."


In [17]:
# 1. Load data
df = load_data(csv_path, max_rows=400)


Loaded 400 rows


In [18]:
# 2. Chunk
chunks, meta = build_chunks(df)


Built 476 chunks


In [20]:
# 3. Embedding model
embedding_model = load_embedding_model()
embeddings = embed_chunks(embedding_model, chunks)


Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding model ready.


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Embeddings shape: (476, 384)


In [21]:
# 4. FAISS index
faiss_index = build_faiss_index(embeddings)


FAISS index built. N=476


In [22]:
# 5. Run query (demo)
query = "Which movie features an evil artificial intelligence?"

result_json = run_rag(query, df, embedding_model, faiss_index, meta, top_k=3)


Query: Which movie features an evil artificial intelligence?


,title,chunk
0,The High Sign,Buster plays a drifter who cons his way into w...
1,Youth's Endearing Charm,The film is about a court case and embezzlement.
2,Dr. Jekyll and Mr. Hyde,White-haired Dr. Jekyll has secretly locked hi...


In [23]:
# 6. Evaluate
print("\nEvaluation metrics:")
display(evaluate_rag(embedding_model, faiss_index, meta))


Evaluation metrics:


,query,hit_rate,max_similarity
0,Which movie has an AI that becomes dangerous?,True,0.3874
